# 감정 분류 모델 학습 (AI-Hub + KOTE 통합)

**목표**: AI-Hub(성인+청소년) + KOTE 데이터를 합쳐서 7종 감정 분류 모델 학습

**라벨**: 기쁨, 없음, 놀라움, 사랑스러움, 화남, 두려움, 슬픔

**흐름**
1. 라이브러리 불러오기
2. AI-Hub JSON 로드
3. KOTE tsv 로드 + 라벨 매핑
4. 두 데이터셋 합치기 + 저장
5. 라벨 분포 확인
6. 언더샘플링 (클래스 균형)
7. TF-IDF 벡터화 + 학습/테스트 분할
8. 모델 학습 (로지스틱 회귀)
9. 평가 (혼동행렬)
10. 모델 저장
11. 빠른 테스트

**주의**: 이 노트북은 위에서부터 순서대로 한 번에 쭉 실행하는 용도입니다.
중간에 셀을 건너뛰거나 순서를 바꾸면 오류가 날 수 있습니다.
셀을 수정했다면 반드시 "커널 재시작 + 모두 실행"으로 처음부터 다시 돌리세요.


## 1. 라이브러리 불러오기

In [1]:
import json
import glob
import os
import pickle

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import resample


## 2. AI-Hub 데이터 로드

`VerifyEmotionTarget`을 원본 라벨로 사용 (검수자가 확인한 값이라 노이즈가 적음)


In [2]:
AIHUB_DATA_DIRS = [
    "data/성인라벨링데이터",
    "data/청소년라벨링데이터",
]  # 본인 폴더 구조에 맞게 수정

def load_aihub_json(data_dirs):
    records = []
    json_files = []
    for d in data_dirs:
        found = glob.glob(os.path.join(d, "**", "*.json"), recursive=True)
        print(f"{d}: {len(found)}개 파일 발견")
        json_files += found

    print(f"AI-Hub 전체 JSON 파일 수: {len(json_files)}")

    for filepath in json_files:
        try:
            with open(filepath, encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            print(f"파일 읽기 실패: {filepath} ({e})")
            continue

        for turn in data.get("Conversation", []):
            text = turn.get("Text", "").strip()
            emotion = turn.get("VerifyEmotionTarget", "").strip()
            if not text or not emotion:
                continue
            records.append({"text": text, "emotion": emotion, "source": "aihub"})

    return pd.DataFrame(records)

df_aihub = load_aihub_json(AIHUB_DATA_DIRS)
print(f"AI-Hub 문장 수: {len(df_aihub)}")
print(df_aihub["emotion"].value_counts())


data/성인라벨링데이터: 90개 파일 발견
data/청소년라벨링데이터: 94개 파일 발견
AI-Hub 전체 JSON 파일 수: 184
AI-Hub 문장 수: 54747
emotion
기쁨       23044
없음       18995
놀라움       6000
사랑스러움     1910
화남        1647
두려움       1619
슬픔        1532
Name: count, dtype: int64


## 3. KOTE 데이터 로드 + 라벨 매핑

KOTE는 44개 세분화 감정이 멀티라벨로 붙어있음.
우리 7종 라벨(기쁨/없음/놀라움/사랑스러움/화남/두려움/슬픔)에 대응되는 것만 매핑하고,
애매한 라벨(기대감/의심불신/비장함 등)은 매핑 테이블에서 제외 -> 자동으로 스킵됨.

`data/train.tsv`, `data/test.tsv`, `data/val.tsv` 파일이 있어야 함 (헤더 없음, 컬럼: id, text, labels)


In [3]:
# 44개 라벨 이름 (KOTE 원본 인덱스 순서 그대로)
kote_label_names = ['불평/불만', '환영/호의', '감동/감탄', '지긋지긋', '고마움', '슬픔', '화남/분노', '존경',
                     '기대감', '우쭐댐/무시함', '안타까움/실망', '비장함', '의심/불신', '뿌듯함', '편안/쾌적',
                     '신기함/관심', '아껴주는', '부끄러움', '공포/무서움', '절망', '한심함', '역겨움/징그러움',
                     '짜증', '어이없음', '없음', '패배/자기혐오', '귀찮음', '힘듦/지침', '즐거움/신남', '깨달음',
                     '죄책감', '증오/혐오', '흐뭇함(귀여움/예쁨)', '당황/난처', '경악', '부담/안_내킴', '서러움',
                     '재미없음', '불쌍함/연민', '놀람', '행복', '불안/걱정', '기쁨', '안심/신뢰']

assert len(kote_label_names) == 44, "라벨 개수가 44개가 아닙니다. 리스트 확인 필요"

# 우리 7종 라벨 매핑 테이블
kote_to_ours = {
    "기쁨": "기쁨", "행복": "기쁨", "즐거움/신남": "기쁨", "뿌듯함": "기쁨",
    "흐뭇함(귀여움/예쁨)": "기쁨", "감동/감탄": "기쁨",
    "없음": "없음",
    "놀람": "놀라움", "경악": "놀라움", "당황/난처": "놀라움", "신기함/관심": "놀라움",
    "아껴주는": "사랑스러움", "고마움": "사랑스러움", "존경": "사랑스러움",
    "안심/신뢰": "사랑스러움", "환영/호의": "사랑스러움", "편안/쾌적": "사랑스러움",
    "화남/분노": "화남", "짜증": "화남", "지긋지긋": "화남", "역겨움/징그러움": "화남",
    "증오/혐오": "화남", "어이없음": "화남", "한심함": "화남", "우쭐댐/무시함": "화남",
    "공포/무서움": "두려움", "불안/걱정": "두려움", "부담/안_내킴": "두려움",
    "슬픔": "슬픔", "절망": "슬픔", "서러움": "슬픔", "패배/자기혐오": "슬픔",
    "죄책감": "슬픔", "힘듦/지침": "슬픔", "안타까움/실망": "슬픔", "불쌍함/연민": "슬픔",
}

kote_files = ["data/train.tsv", "data/test.tsv", "data/val.tsv"]
df_kote_raw = pd.concat([
    pd.read_csv(f, sep="\t", header=None, names=["id", "text", "labels"])
    for f in kote_files
], ignore_index=True)

print(f"KOTE 원본 문장 수: {len(df_kote_raw)}")

records = []
for _, row in df_kote_raw.iterrows():
    text = str(row["text"]).strip()
    if not text or pd.isna(row["labels"]):
        continue

    idxs = [int(x) for x in str(row["labels"]).split(",")]
    mapped = set()
    for idx in idxs:
        if idx < len(kote_label_names):
            name = kote_label_names[idx]
            if name in kote_to_ours:
                mapped.add(kote_to_ours[name])

    for emo in mapped:
        records.append({"text": text, "emotion": emo, "source": "kote"})

df_kote = pd.DataFrame(records)
print(f"KOTE 매핑 후 문장 수: {len(df_kote)}")
print(df_kote["emotion"].value_counts())


KOTE 원본 문장 수: 50000
KOTE 매핑 후 문장 수: 148528
emotion
화남       29939
놀라움      28522
슬픔       26677
기쁨       21529
사랑스러움    20399
두려움      13624
없음        7838
Name: count, dtype: int64


## 4. 두 데이터셋 합치기 + 저장

In [4]:
df = pd.concat([df_aihub, df_kote], ignore_index=True)
df = df.drop_duplicates(subset=["text", "emotion"]).reset_index(drop=True)

print(f"합친 후 총 문장 수: {len(df)}")
print()
print("[출처별 문장 수]")
print(df["source"].value_counts())

os.makedirs("data", exist_ok=True)
df.to_csv("data/emotion_dataset_merged.csv", index=False, encoding="utf-8-sig")
print("\n저장 완료: data/emotion_dataset_merged.csv")


합친 후 총 문장 수: 192266

[출처별 문장 수]
source
kote     148528
aihub     43738
Name: count, dtype: int64

저장 완료: data/emotion_dataset_merged.csv


## 5. 라벨 분포 확인

(다음부터는 항상 `data/emotion_dataset_merged.csv`를 다시 불러오는 걸 기준으로 진행 -> 재실행 시에도 위 2~4번을 건너뛰고 아래 셀부터 바로 시작 가능)


In [5]:
df = pd.read_csv("data/emotion_dataset_merged.csv")
print(f"전체 문장 수: {len(df)}")
print(df["emotion"].value_counts())


전체 문장 수: 192266
emotion
기쁨       41342
놀라움      34025
화남       31512
슬픔       28136
사랑스러움    21991
없음       20185
두려움      15075
Name: count, dtype: int64


## 6. 언더샘플링 (클래스 균형 맞추기)

7종 라벨 4종으로 통합하고 예외를 추가, 각 클래스 데이터 수를 가장 적은 클래스 기준으로 맞춤


In [6]:
# "없음" 제외, 4종으로 통합
emotion_map = {
    "기쁨": "기쁨",
    "사랑스러움": "기쁨",
    "슬픔": "슬픔",
    "화남": "화남",
    "두려움": "두려움/놀라움",
    "놀라움": "두려움/놀라움",
}
df_train_ready = df[df["emotion"] != "없음"].copy()
df_train_ready["emotion"] = df_train_ready["emotion"].map(emotion_map)

print(df_train_ready["emotion"].value_counts())

emotion
기쁨         63333
두려움/놀라움    49100
화남         31512
슬픔         28136
Name: count, dtype: int64


In [7]:
min_count = df_train_ready["emotion"].value_counts().min()

df_balanced = pd.concat([
    resample(df_train_ready[df_train_ready["emotion"] == label],
             n_samples=min_count, random_state=42)
    for label in df_train_ready["emotion"].unique()
])
print(df_balanced["emotion"].value_counts())

emotion
두려움/놀라움    28136
기쁨         28136
화남         28136
슬픔         28136
Name: count, dtype: int64


## 7. 학습/테스트 분할 + TF-IDF 벡터화

In [8]:
from konlpy.tag import Okt
okt = Okt()

def tokenize(text):
    return okt.morphs(text)

X_train, X_test, y_train, y_test = train_test_split(
    df_balanced["text"], df_balanced["emotion"],
    test_size=0.2, random_state=42, stratify=df_balanced["emotion"]
)

vectorizer = TfidfVectorizer(tokenizer=tokenize, max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"학습 데이터: {X_train_vec.shape}, 테스트 데이터: {X_test_vec.shape}")


c:\AI_projact\venv\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


학습 데이터: (90035, 5000), 테스트 데이터: (22509, 5000)


## 8. 모델 학습 (baseline: 로지스틱 회귀)

In [9]:
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

          기쁨       0.50      0.63      0.56      5627
     두려움/놀라움       0.31      0.24      0.27      5627
          슬픔       0.36      0.33      0.34      5628
          화남       0.39      0.40      0.39      5627

    accuracy                           0.40     22509
   macro avg       0.39      0.40      0.39     22509
weighted avg       0.39      0.40      0.39     22509



## 9. 혼동행렬 확인

In [10]:
labels_sorted = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels_sorted)
cm_df = pd.DataFrame(cm, index=labels_sorted, columns=labels_sorted)
cm_df


,기쁨,두려움/놀라움,슬픔,화남
기쁨,3539,926,646,516
두려움/놀라움,1682,1371,1232,1342
슬픔,994,1083,1860,1691
화남,830,1074,1472,2251


## 10. 모델 저장

In [11]:
with open("emotion_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("모델 저장 완료: emotion_model.pkl, vectorizer.pkl")


모델 저장 완료: emotion_model.pkl, vectorizer.pkl


## 11. 빠른 테스트

In [14]:
def predict_emotion(text, threshold=0.35):
    vec = vectorizer.transform([text])
    proba = model.predict_proba(vec)[0]
    prob_dict = dict(zip(model.classes_, proba))

    max_prob = max(proba)
    if max_prob < threshold:
        pred = "없음"
    else:
        pred = model.classes_[proba.argmax()]

    return pred, prob_dict

sample_text = "슬프네"
pred, probs = predict_emotion(sample_text)
print(f"입력: {sample_text}")
print(f"예측 감정: {pred}")
print(f"확률 분포: {probs}")

입력: 슬프네
예측 감정: 기쁨
확률 분포: {'기쁨': np.float64(0.4567064218541269), '두려움/놀라움': np.float64(0.2438834726204445), '슬픔': np.float64(0.14024428921770207), '화남': np.float64(0.15916581630772647)}


In [13]:
print("[언더샘플링 후 df_balanced 분포]")
print(df_balanced["emotion"].value_counts())
print()
print("[벡터라이저 설정 확인]")
print(vectorizer.analyzer, vectorizer.ngram_range)

[언더샘플링 후 df_balanced 분포]
emotion
두려움/놀라움    28136
기쁨         28136
화남         28136
슬픔         28136
Name: count, dtype: int64

[벡터라이저 설정 확인]
word (1, 1)
